# DeepNoise: Animal Sound Classification Master Pipeline

Welcome to the master pipeline notebook for the **Environmental Sound (Animal Sound) Classification** subproject. This notebook walks through the entire pipeline: data ingestion, preprocessing, offline data augmentation, baseline models, CNN training, and model evaluation.

### Pipeline Structure
The workflow consists of the following steps:
1. **`download_data.py`**: Downloads raw environmental WAV files from ESC-50 and extracts the target 8 animal classes.
2. **`preprocess.py`**: Standardizes audio to 22,050 Hz and 5.0 seconds, applies 6 offline data augmentations (yielding 1,920+ training samples), and extracts Log-Mel Spectrogram features.
3. **`baseline_model.py`**: Evaluates traditional ML baselines (Random Forest & Support Vector Machine) on statistical features.
4. **`train.py`**: Trains our custom lightweight 2D CNN model in PyTorch on Folds 1-3, validating on Fold 4, and saving the best checkpoint.
5. **`evaluate.py`**: Evaluates the trained CNN on the unseen test set (Fold 5), generating classification reports and a confusion matrix heatmap.

---

## Step 1: Ingest Raw Dataset

We start by running the data downloader script to fetch the ESC-50 dataset ZIP archive, extract the metadata, and extract the WAV recordings for the target 8 animal classes (`dog`, `rooster`, `pig`, `cow`, `frog`, `cat`, `sheep`, `hen`).

In [ ]:
%run download_data.py

## Step 2: Audio Preprocessing & Data Augmentation

Next, we run the preprocessor. This script standardizes the raw WAV files to 22,050 Hz and 5.0 seconds, and generates 6 augmented variants per clip (pitch shift up/down, speed up/down, time shift, and Gaussian noise). Finally, it extracts Log-Mel Spectrogram representations and saves them as `.npy` feature files.

In [ ]:
%run preprocess.py

## Step 3: Traditional ML Baselines

With the extracted Mel-spectrogram features ready, we compute their statistical summaries (mean and standard deviation for each Mel band) to create a 256-dimensional feature representation. We train both a **Random Forest** and a **Support Vector Machine (SVM)** classifier, compare them on the validation fold (Fold 4), and evaluate the winner on the test split (Fold 5).

In [ ]:
%run baseline_model.py

## Step 4: CNN Training (Native PyTorch)

Now we train our custom 2D CNN model in native PyTorch. The training script:
- Loads features from Folds 1-3 (Train) and Fold 4 (Validation).
- Sets up PyTorch DataLoaders with a batch size of 16.
- Runs training on the GPU (RTX 3050 Ti).
- Employs early stopping on validation loss to prevent overfitting.
- Saves optimal checkpoint weights to `models/best_cnn.pth` and test features to `models/test_split.npz`.

In [ ]:
%run train.py

## Step 5: CNN Model Evaluation

Finally, we run the evaluation script to test the CNN's accuracy, precision, recall, and F1-score on the unseen test fold (Fold 5). It prints the per-class classification report and saves the Confusion Matrix heatmap to `results/cnn_confusion_matrix.png`.

In [ ]:
%run evaluate.py